<a href="https://colab.research.google.com/github/WalidYaser/Deep-Learning/blob/main/Transformer_Encoder_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Transformer Encoder Assignment — COMPLETE SOLUTION
Sentiment Analysis Model from Scratch using PyTorch
"""

# ─────────────────────────────────────────────
# Imports and Setup
# ─────────────────────────────────────────────
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


# ─────────────────────────────────────────────
# 1. Dataset
# ─────────────────────────────────────────────
texts = [
    "i love this movie",
    "this film is amazing",
    "what a wonderful story",
    "great acting and plot",
    "i really enjoyed it",
    "best film i have seen",
    "absolutely fantastic experience",
    "i hate this movie",
    "what a terrible film",
    "boring and disappointing",
    "worst movie ever made",
    "i did not enjoy it",
    "awful acting and weak plot",
    "completely dreadful experience",
]
labels = [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0]


# ─────────────────────────────────────────────
# 2. Build Vocabulary
# ─────────────────────────────────────────────
def build_vocab(texts):
    vocab = {"<pad>": 0}
    for text in texts:
        for word in text.split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

vocab = build_vocab(texts)
print(f"Vocabulary size: {len(vocab)}")


# ─────────────────────────────────────────────
# TODO 1 ✅ — encode_text
# Convert a text string into a list of integer indices based on vocab.
# Unknown words → 0 (<pad> index).
# ─────────────────────────────────────────────
def encode_text(text, vocab):
    return [vocab.get(word, 0) for word in text.split()]


# ─────────────────────────────────────────────
# TODO 2 ✅ — pad_sequence
# Ensure the sequence is exactly max_len long:
#   shorter → pad with pad_value at the end
#   longer  → truncate to max_len
# ─────────────────────────────────────────────
def pad_sequence(seq, max_len, pad_value=0):
    if len(seq) < max_len:
        seq = seq + [pad_value] * (max_len - len(seq))   # pad
    else:
        seq = seq[:max_len]                               # truncate
    return seq


# ─────────────────────────────────────────────
# 5. Dataset Preparation
# ─────────────────────────────────────────────
MAX_LEN = 10
PAD_IDX = 0

encoded   = [pad_sequence(encode_text(t, vocab), MAX_LEN) for t in texts]
X         = torch.tensor(encoded, dtype=torch.long)
y         = torch.tensor(labels,  dtype=torch.long)

dataset   = TensorDataset(X, y)
loader    = DataLoader(dataset, batch_size=4, shuffle=True)


# ─────────────────────────────────────────────
# 6. Transformer Encoder Model
# TODO 3, 4, 5 ✅
# ─────────────────────────────────────────────
class TransformerSentimentClassifier(nn.Module):
    def __init__(self, vocab_size, input_size, num_heads, num_layers,
                 num_classes, max_len, pad_idx):
        super().__init__()

        # TODO 3.1 ✅ — Word Embedding layer
        # vocab_size tokens mapped to input_size-dimensional vectors.
        # padding_idx=pad_idx ensures the <pad> token always has a zero
        # gradient and a zero embedding so it never influences learning.
        self.embedding = nn.Embedding(vocab_size, input_size,
                                      padding_idx=pad_idx)

        # TODO 3.2 ✅ — Positional Embedding layer
        # One learnable vector per position (0 … max_len-1), same dimension
        # as the word embedding so they can be summed directly.
        self.pos_embedding = nn.Embedding(max_len, input_size)

        # TODO 4 ✅ — Single Transformer Encoder Layer
        # d_model   : dimension of the model (must equal input_size)
        # nhead     : number of self-attention heads
        # dim_feedforward : width of the inner FFN (128 as specified)
        # batch_first=True : input/output shape is (batch, seq, feature)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=input_size,
            nhead=num_heads,
            dim_feedforward=128,
            dropout=0.1,
            batch_first=True
        )

        # TODO 5 ✅ — Stack encoder layers
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.fc = nn.Linear(input_size, num_classes)

    def forward(self, x):
        batch_size, seq_len = x.shape

        # Word embeddings
        x = self.embedding(x)                                     # (B, T, D)

        # Positional embeddings
        positions = (torch.arange(0, seq_len, device=x.device)
                         .unsqueeze(0)
                         .expand(batch_size, seq_len))            # (B, T)
        pos = self.pos_embedding(positions)                       # (B, T, D)

        x = x + pos                                               # (B, T, D)

        # Transformer encoder
        out = self.transformer_encoder(x)                         # (B, T, D)

        # Take the last token's representation for classification
        out = out[:, -1, :]                                       # (B, D)

        out = self.fc(out)                                        # (B, num_classes)
        return out


# ─────────────────────────────────────────────
# Hyperparameters & Instantiation
# ─────────────────────────────────────────────
VOCAB_SIZE  = len(vocab)
INPUT_SIZE  = 32        # embedding / d_model dimension
NUM_HEADS   = 4         # must divide INPUT_SIZE evenly (32 / 4 = 8)
NUM_LAYERS  = 2
NUM_CLASSES = 2         # Positive / Negative
EPOCHS      = 20
LR          = 1e-3

model = TransformerSentimentClassifier(
    vocab_size=VOCAB_SIZE,
    input_size=INPUT_SIZE,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    num_classes=NUM_CLASSES,
    max_len=MAX_LEN,
    pad_idx=PAD_IDX
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)


# ─────────────────────────────────────────────
# Training Loop
# ─────────────────────────────────────────────
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, correct = 0.0, 0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss   = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        correct    += (logits.argmax(1) == yb).sum().item()

    if epoch % 5 == 0:
        acc = correct / len(dataset) * 100
        print(f"Epoch {epoch:02d} | Loss: {total_loss/len(dataset):.4f} | Acc: {acc:.1f}%")


# ─────────────────────────────────────────────
# TODO 6 ✅ — Inference Function
# Takes a raw text string, preprocesses it, returns "Positive" or "Negative".
# ─────────────────────────────────────────────
def predict_sentiment(model, text, vocab, max_len, device):
    model.eval()
    with torch.no_grad():
        # 1. Encode the raw text into token indices
        encoded = encode_text(text, vocab)
        # 2. Pad / truncate to max_len
        padded  = pad_sequence(encoded, max_len)
        # 3. Convert to tensor and add batch dimension → (1, max_len)
        tensor  = torch.tensor([padded], dtype=torch.long).to(device)
        # 4. Forward pass
        logits  = model(tensor)                    # (1, num_classes)
        # 5. Argmax → class index
        pred    = logits.argmax(dim=1).item()
        return "Positive" if pred == 1 else "Negative"


# ─────────────────────────────────────────────
# Quick demo
# ─────────────────────────────────────────────
test_sentences = [
    "i love this amazing film",
    "this was a terrible and boring movie",
    "what a wonderful experience",
    "i hate this awful story",
]

print("\n--- Inference Demo ---")
for sent in test_sentences:
    print(f"  '{sent}'  →  {predict_sentiment(model, sent, vocab, MAX_LEN, device)}")

Using device: cuda
Vocabulary size: 39
Epoch 05 | Loss: 0.6719 | Acc: 64.3%
Epoch 10 | Loss: 0.5719 | Acc: 71.4%
Epoch 15 | Loss: 0.3038 | Acc: 92.9%
Epoch 20 | Loss: 0.1181 | Acc: 92.9%

--- Inference Demo ---
  'i love this amazing film'  →  Positive
  'this was a terrible and boring movie'  →  Negative
  'what a wonderful experience'  →  Positive
  'i hate this awful story'  →  Negative
